# Session 30a — Prompt Engineering · Hands-on Notebook
### Computer Vision & AI series

This notebook is the practical half of **Session 30a**. It mirrors the slides and lets you *run* every prompt-engineering technique against a real model using the **OpenAI API through LangChain**.

You will practice the four levers from the slides:

| # | Technique | What you'll do |
|---|-----------|----------------|
| 1 | **System prompts** | Change the assistant's persona & rules and watch the answer change |
| 2 | **Few-shot** | Teach a labelling pattern with a handful of examples |
| 3 | **Chain-of-Thought** | Compare a direct answer vs step-by-step reasoning |
| 4 | **Structured output** | Force clean, typed JSON your code can trust |

> **Prerequisite:** an OpenAI API key. Everything else installs in the first cell.

## 0 · Setup

We install LangChain's core plus the OpenAI integration and Pydantic (for structured output).

| Package | Gives us |
|---|---|
| `langchain-core` | prompts, messages, the Runnable interface |
| `langchain-openai` | the `ChatOpenAI` chat model |
| `pydantic` | typed schemas for guaranteed-shape output |

In [1]:
# Run once. Quiet install of everything this notebook needs.
%pip install -q -U langchain-core langchain-openai pydantic
print("Setup complete.")

Note: you may need to restart the kernel to use updated packages.
Setup complete.


### Provide your API key

The key is read from the `OPENAI_API_KEY` environment variable. The cell below asks for it securely (nothing is printed) if it isn't already set.

In [3]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("Key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

Key loaded: True


In [16]:
# One shared model for the whole notebook.
# temperature=0 keeps answers deterministic so teaching results are reproducible.
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Quick smoke test — confirms the key + network work.
print(model.invoke("Reply with exactly: ready").content)

ready


---
# 1 · System Prompts — set the frame

The **system** message defines *who the assistant is* and *how it must behave*: persona, tone, scope, format. It's sent once and silently steers every reply.

A chat request is just a list of `(role, content)` messages. LangChain lets us write that list as tuples.

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

question = "How do I resize an image in Python?"

# Same question, two different system prompts:
terse = [
    SystemMessage("You are a terse senior engineer. Answer in one line of code, no prose."),
    HumanMessage(question),
]
teacher = [
    SystemMessage("You are a patient beginner tutor. Explain each step simply before the code."),
    HumanMessage(question),
]

print("=== TERSE ENGINEER ===\n", model.invoke(terse).content)
print("\n=== PATIENT TUTOR ===\n", model.invoke(teacher).content)

=== TERSE ENGINEER ===
 ```python
from PIL import Image; img = Image.open('image.jpg'); img = img.resize((width, height)); img.save('resized_image.jpg')
```

=== PATIENT TUTOR ===
 Resizing an image in Python can be done using various libraries, but one of the most popular and user-friendly libraries for image processing is Pillow (PIL). Below, I'll guide you through the steps to resize an image using Pillow.

### Step 1: Install Pillow
First, you need to make sure you have the Pillow library installed. You can install it using pip. Open your command line or terminal and run:

```bash
pip install Pillow
```

### Step 2: Import the Library
Once Pillow is installed, you need to import it in your Python script. You will specifically import the `Image` class from the `PIL` module.

```python
from PIL import Image
```

### Step 3: Open the Image
Next, you need to open the image you want to resize. You can do this by using the `Image.open()` method, which takes the file path of the image as 

**What just happened?** The user's question never changed — only the *system prompt* did. That single message reshaped the tone, length, and structure of the answer. This is your first and most powerful lever.

**Best practice:** put stable rules (persona, format, safety limits) in the system prompt, and state them *positively* — say what the model **should** do, not only what to avoid.

### Reusable system prompts with a template

Hard-coding the wording is brittle. A `ChatPromptTemplate` is a reusable recipe with blanks you fill at runtime — the same idea you saw with prompt templates in Session 29.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

tutor_prompt = ChatPromptTemplate([
    ("system", "You are VisionTutor, a concise computer-vision teacher. "
               "Explain with one concrete image example. Keep answers under 60 words."),
    ("human", "{question}"),
])

# Chain the template into the model with the pipe operator (LCEL).
chain = tutor_prompt | model

print(chain.invoke({"question": "What is a convolution?"}).content)

Convolution is a mathematical operation used in image processing. For example, consider a 3x3 filter (kernel) applied to a 5x5 image. The filter slides over the image, multiplying its values with the overlapping pixel values, summing them up to produce a new pixel value in the output image. This helps in feature extraction, like edge detection.


---
# 2 · Few-Shot Prompting — teach by example

For custom formats or nuanced judgements, *showing* beats *telling*. We put a few worked **examples** in the prompt; the model infers the pattern and continues it. This is **in-context learning** (Brown et al., 2020).

First, a **zero-shot** attempt at a custom labelling task — notice the output format is unpredictable.

In [7]:
zero_shot = [
    SystemMessage("Classify the customer message intent."),
    HumanMessage("The app crashes every time I upload a photo."),
]
print("ZERO-SHOT:", model.invoke(zero_shot).content)

ZERO-SHOT: The customer message intent is to report a technical issue or bug with the app.


Now the **few-shot** version. We give three labelled examples in our exact target vocabulary (`bug` / `how-to` / `praise`), then the real input. `AIMessage` represents the *assistant's* turn — that's how we show the model a completed example, so it locks onto the pattern.

In [8]:
from langchain_core.messages import AIMessage

few_shot = [
    SystemMessage("Label each customer message as exactly one of: bug | how-to | praise. "
                  "Reply with only the label, lowercase."),
    # --- worked examples ---
    HumanMessage("It keeps logging me out randomly."),          AIMessage("bug"),
    HumanMessage("Where do I change my password?"),             AIMessage("how-to"),
    HumanMessage("Honestly the best editing app I've used!"),   AIMessage("praise"),
    # --- the real input ---
    HumanMessage("The app crashes every time I upload a photo."),
]

print("FEW-SHOT:", model.invoke(few_shot).content)

FEW-SHOT: bug


The output is now a single clean label — exactly the vocabulary we demonstrated. Few-shot examples act as a *contract* for the format.

**Try it:** run the batch below to label several new messages at once.

In [9]:
new_messages = [
    "How do I export to PNG?",
    "Love the new dark mode!",
    "Filters stopped working after the update.",
]

def label(msg):
    convo = [
        SystemMessage("Label each customer message as exactly one of: bug | how-to | praise. "
                      "Reply with only the label, lowercase."),
        HumanMessage("It keeps logging me out randomly."),         AIMessage("bug"),
        HumanMessage("Where do I change my password?"),            AIMessage("how-to"),
        HumanMessage("Honestly the best editing app I've used!"),  AIMessage("praise"),
        HumanMessage(msg),
    ]
    return model.invoke(convo).content.strip()

for m in new_messages:
    print(f"{label(m):8} <- {m}")

how-to   <- How do I export to PNG?
praise   <- Love the new dark mode!
bug      <- Filters stopped working after the update.


---
# 3 · Chain-of-Thought — ask for reasoning

On multi-step or arithmetic questions, asking the model to **show its work** before answering sharply improves accuracy (Wei et al., 2022). Below, the *same* problem is asked two ways.

In [10]:
problem = ("A shop had 23 apples. It used 20 to make pies, then a delivery "
           "brought 6 more boxes of 4 apples each. How many apples now?")

# (a) Direct answer — no reasoning encouraged
direct = [
    SystemMessage("Answer with only the final number, nothing else."),
    HumanMessage(problem),
]
print("DIRECT:", model.invoke(direct).content)

DIRECT: 27


Now **zero-shot Chain-of-Thought**: we simply add *“Let's think step by step.”* — the single phrase from Kojima et al. (2022) that unlocks reasoning without any examples.

In [11]:
cot = [
    SystemMessage("Solve carefully. Let's think step by step, then give the final answer."),
    HumanMessage(problem),
]
print(model.invoke(cot).content)

Let's break down the problem step by step:

1. **Initial number of apples**: The shop starts with 23 apples.

2. **Apples used for pies**: The shop uses 20 apples to make pies. 
   - After using 20 apples, the number of apples left is:
     \[
     23 - 20 = 3 \text{ apples}
     \]

3. **Delivery of new apples**: The shop receives a delivery of 6 boxes, with each box containing 4 apples. 
   - First, we calculate the total number of apples delivered:
     \[
     6 \text{ boxes} \times 4 \text{ apples/box} = 24 \text{ apples}
     \]

4. **Total number of apples after delivery**: Now, we add the apples left after making pies to the apples received from the delivery:
   \[
   3 \text{ apples} + 24 \text{ apples} = 27 \text{ apples}
   \]

Thus, the final number of apples in the shop is **27 apples**.


The step-by-step version walks through `23 − 20 = 3`, then `6 × 4 = 24`, then `3 + 24 = 27` — and lands on the right answer. Reasoning out loud catches mistakes a snap answer misses.

**When to use it:** maths, logic, planning, multi-step questions. **Skip it** for simple lookups — it costs extra tokens and latency.

> Tip: to keep a *clean* final answer while still benefiting from reasoning, ask the model to reason and then end with a line like `Final answer: <x>`, and parse that line.

---
# 4 · Structured Output — text → reliable data

Apps need **fields, not paragraphs**. Instead of parsing free text, we hand the model a **schema** and it returns data guaranteed to match it. With LangChain + OpenAI, the cleanest way is `with_structured_output()` backed by a **Pydantic** model.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Review(BaseModel):
    """Structured analysis of a product review."""
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="overall sentiment")
    score: float = Field(description="confidence from 0 to 1")
    topics: list[str] = Field(description="key topics mentioned, e.g. delivery, price")

# Bind the schema to the model — output is now guaranteed to be a Review object.
extractor = model.with_structured_output(Review)

result = extractor.invoke("The camera quality is amazing but delivery took three weeks.")
print(type(result))
print(result)

<class '__main__.Review'>
sentiment='neutral' score=0.7 topics=['camera quality', 'delivery time']


`result` is a **typed Python object**, not a string. We can use its fields directly — no fragile string-splitting, no `json.loads` guesswork.

In [17]:
print("sentiment:", result.sentiment)
print("score    :", result.score)
print("topics   :", result.topics)

# It's real data — branch on it like any object:
if result.sentiment == "negative" or result.score < 0.5:
    print("\n-> route to a human agent")
else:
    print("\n-> auto-acknowledge")

sentiment: neutral
score    : 0.7
topics   : ['camera quality', 'delivery time']

-> auto-acknowledge


### Batch extraction

Because the extractor is a Runnable, `.batch()` runs many inputs efficiently and returns a list of typed objects.

In [14]:
reviews = [
    "Fast shipping and the lens is razor sharp. Five stars!",
    "Battery dies in an hour. Very disappointed.",
    "It's fine. Does the job, nothing special.",
]

for r in extractor.batch(reviews):
    print(f"{r.sentiment:8} ({r.score:.2f})  topics={r.topics}")

positive (0.95)  topics=['shipping', 'lens quality', 'customer satisfaction']
negative (0.90)  topics=['battery life', 'disappointment', 'product performance']
neutral  (0.50)  topics=['performance', 'quality', 'value']


---
# 5 · Putting it together

A single request can stack **all four techniques**: a *system prompt* sets the role and asks for *Chain-of-Thought*, *few-shot* examples fix the style, and *structured output* guarantees the shape. Below we build a support-ticket triager.

In [15]:
from typing import Literal
from pydantic import BaseModel, Field

class Ticket(BaseModel):
    """Triage result for a support ticket."""
    category: Literal["bug", "billing", "how-to", "feedback"]
    priority: Literal["low", "medium", "high"]
    reason: str = Field(description="one short sentence justifying the priority")

triager = ChatPromptTemplate([
    ("system",
     "You are a support-ticket triager. Think about the impact before deciding, "
     "then classify the ticket. Follow the style of the examples."),
    # few-shot examples (as a mini dialogue)
    ("human", "App crashes on every upload."),
    ("ai", "category=bug, priority=high, reason=Blocks a core action for the user."),
    ("human", "I was charged twice this month."),
    ("ai", "category=billing, priority=high, reason=Financial impact needs fast resolution."),
    # real ticket
    ("human", "{ticket}"),
]) | model.with_structured_output(Ticket)

out = triager.invoke({"ticket": "How do I turn on dark mode?"})
print(out)

category='how-to' priority='low' reason='User needs assistance with a feature.'


Notice how each lever contributed: the **system prompt** set the role and asked for reasoning, the **few-shot** pairs fixed the vocabulary and style, and **structured output** returned a typed `Ticket` — ready to drop into a database or route automatically.

---
## Recap

| Lever | One-line takeaway |
|---|---|
| **System prompt** | Sets role, rules and format once — the frame for every reply. |
| **Few-shot** | Show examples to teach a pattern; keep the format identical. |
| **Chain-of-Thought** | “Let's think step by step” for reasoning-heavy tasks. |
| **Structured output** | Bind a Pydantic schema so text becomes reliable, typed data. |

**Next → Session 30b: Chatbot Architecture** — how these prompts live inside a stateful, multi-turn conversation with memory.

### References
- Brown et al. (2020). *Language Models are Few-Shot Learners* (GPT-3). arXiv:2005.14165
- Wei et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in LLMs.* arXiv:2201.11903
- Kojima et al. (2022). *Large Language Models are Zero-Shot Reasoners.* arXiv:2205.11916
- OpenAI. *Introducing Structured Outputs in the API* (2024).
- LangChain docs — *Structured output* & *ChatOpenAI*.